# Tine: Step 1 graded data, CPU baseline, optional embeddings

Start with a **CPU runtime**. Run setup, Section 1 (fetch and import), and Section 2 (data preflight). The preflight is the end of Step 1. It shows whether the selected benchmark has enough independent risk-calibration tasks per group before you use a GPU.

The archive is about 1.28 GB compressed. It contains already-graded results. This notebook runs zero model-generation or provider API calls. Section 3 optionally produces a lexical CPU baseline. Sections 4–5 download a small encoder and are optional later work.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import subprocess
import sys
import json

ROOT = Path("/content/drive/MyDrive/tine")
ROOT.mkdir(parents=True, exist_ok=True)
REPO = Path("/content/tine")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/Nandu-Dammalapti/tine.git", str(REPO)], check=True)
os.chdir(REPO)
if not (REPO / "pyproject.toml").is_file():
    raise RuntimeError("Extract the repository so /content/tine/pyproject.toml exists, then rerun this cell.")

def repository_commit():
    result = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True)
    return result.stdout.strip() if result.returncode == 0 else None

print("Repository commit:", repository_commit() or "ZIP source; no Git revision available")
print("Persistent output directory:", ROOT)


If cloning fails, use Colab's GitHub integration or upload the current repository ZIP into `/content/tine`, then rerun the cell. Never paste access tokens into a cell or commit them. The archive and extracted results in Section 1 are saved on Drive, so leave enough Drive space for both.


In [ ]:
if sys.version_info < (3, 12):
    raise RuntimeError("Tine requires Python 3.12 or newer. Choose a current Python runtime before installing.")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "huggingface-hub==0.36.0"], check=True)
os.environ["MPLBACKEND"] = "Agg"
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
subprocess.run([sys.executable, "-m", "tine", "--help"], check=True)


## 1. Fetch and import already-graded LiveCodeBench results on CPU

The helper checks the archive against the pinned Hugging Face LFS SHA-256 when available and records both digests in the manifest. It rejects unsafe archive paths and limits extracted bytes. Review the [dataset and its terms](https://huggingface.co/datasets/NPULH/LLMRouterBench) before use.

If you already have graded canonical Tine outcomes on Drive, skip the next two cells and set `DATASET` in the inspection cell. HumanEval and MBPP task-only files contain prompts and tests for later work. Use a graded outcome file here.


In [ ]:
BENCHMARK_REVISION = "b95870f9e60c589e82f7972c91ffa8de9ca89f85"
fetch_command = [sys.executable, "scripts/fetch_benchmark.py",
                 "--output-dir", str(ROOT / "benchmark"),
                 "--revision", BENCHMARK_REVISION,
                 "--max-extracted-gb", "20"]
with subprocess.Popen(fetch_command, cwd=REPO, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True) as fetch:
    for line in fetch.stdout:
        print(line, end="", flush=True)
    if fetch.wait() != 0:
        raise RuntimeError("Benchmark fetch failed. The specific error is printed above.")

manifest = json.loads((ROOT / "benchmark" / "manifest.json").read_text())


In [ ]:
# Reuse existing binary outcomes and recorded per-task costs.
SOURCE = Path(manifest["bench_dir"])
if (SOURCE / "livecodebench").is_dir():
    SOURCE = SOURCE / "livecodebench"
DATASET = ROOT / "outcomes.jsonl"
subprocess.run([sys.executable, "-m", "tine", "import-llmrouterbench",
                "--input", str(SOURCE), "--output", str(DATASET),
                "--tenant", "benchmark", "--dataset", "livecodebench",
                "--run-selection", "latest"], check=True)


## 2. Inspect outcomes and finish Step 1

Set `FRONTIER` to the exact name of the default model you intend to compare against. Choose it before running evaluation. The preflight uses the same split and default threshold grid as the evaluator and reports how many risk-calibration tasks are needed in the best case of zero observed degradations.


In [ ]:
from tine.data import read_outcomes

DATASET = ROOT / "outcomes.jsonl"  # Change only if using your own canonical file.
outcomes = read_outcomes(DATASET)
print("Rows:", len(outcomes))
print("Tasks:", outcomes[["tenant", "task_id"]].drop_duplicates().shape[0])
print(outcomes.groupby("model").agg(rows=("task_id", "size"), mean_cost_usd=("cost_usd", "mean")).to_string())
print("Available frontier names:", sorted(outcomes["model"].unique().tolist()))


In [ ]:
from tine.pipeline import preflight

FRONTIER = ""  # REQUIRED: copy the exact name of your intended default frontier from the list above.
if FRONTIER not in set(outcomes["model"]):
    raise ValueError("Set FRONTIER to an exact model name printed above before running evaluation.")
EPSILON = 0.02
DELTA = 0.05
SEED = 42

readiness = preflight(outcomes, FRONTIER, epsilon=EPSILON, delta=DELTA, seed=SEED)
(ROOT / "step1-preflight.json").write_text(json.dumps(readiness, indent=2) + "\n")
print("Complete distinct tasks:", readiness["complete_tasks"])
print("Excluded tasks:", len(readiness["excluded"]))
print("Minimum risk tasks per group with zero degradations:",
      readiness["minimum_risk_tasks_at_zero_degradations"])
print(json.dumps(readiness["groups"], indent=2))
if not all(group["enough_risk_tasks_at_zero_degradations"] for group in readiness["groups"]):
    print("Groups below the minimum will use the frontier under the default calibration settings.")

revision = repository_commit()
run_config = {"dataset": str(DATASET), "frontier": FRONTIER, "epsilon": EPSILON,
              "delta": DELTA, "seed": SEED,
              "repository_commit": revision, "source_type": "git" if revision else "zip"}
(ROOT / "run-config.json").write_text(json.dumps(run_config, indent=2) + "\n")
print("Step 1 complete. Keep outcomes.jsonl, step1-preflight.json, and the benchmark manifest on Drive.")


## 3. Optional next stage: run the CPU baseline

You can stop after Section 2. Run this section when you want held-out routing results using model-free lexical features. It uses separate training, score-calibration, risk-calibration, and test splits. Inspect the calibration decisions and data exclusions in the report before moving to embeddings.


In [ ]:
CPU_REPORT = ROOT / "cpu-lexical"
subprocess.run([sys.executable, "-m", "tine", "run", "--input", str(DATASET),
                "--frontier", FRONTIER, "--output", str(CPU_REPORT),
                "--epsilon", str(EPSILON), "--delta", str(DELTA), "--seed", str(SEED)], check=True)
print("CPU artifacts:", sorted(str(p) for p in CPU_REPORT.iterdir()))


## 4. Optional: pin the embedding model and check 50 prompts on CPU

The notebook obtains the encoder's full commit SHA from Hugging Face's official metadata API once, then saves that exact revision to Drive. Reopening this notebook reuses the saved pin. Existing checkpoints reject changed prompts, task order, encoder revision, batch size, or truncation settings.

This encoder truncates each prompt at 256 model tokens. That is a documented first baseline, not a claim that all long-prompt information is preserved. Keep this configuration fixed for a comparison.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install",
                "sentence-transformers==5.1.2", "huggingface-hub==0.36.0"], check=True)
from huggingface_hub import model_info
import re

encoder_config_path = ROOT / "encoder-config.json"
if encoder_config_path.exists():
    encoder_config = json.loads(encoder_config_path.read_text())
else:
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    encoder_config = {"model": model_name, "revision": model_info(model_name).sha,
                      "batch_size": 64, "max_seq_length": 256}
    encoder_config_path.write_text(json.dumps(encoder_config, indent=2) + "\n")
if not re.fullmatch(r"[0-9a-fA-F]{40}", encoder_config["revision"]):
    raise ValueError("A verified full model commit SHA is required")
print("Pinned encoder:", json.dumps(encoder_config, indent=2))
(ROOT / "colab-packages.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))


In [ ]:
embed_base = [sys.executable, "scripts/embed.py", "--input", str(DATASET),
              "--model", encoder_config["model"], "--revision", encoder_config["revision"],
              "--batch-size", str(encoder_config["batch_size"]),
              "--max-seq-length", str(encoder_config["max_seq_length"]),
              "--cache-dir", "/content/tine-model-cache"]
subprocess.run(embed_base + ["--output", str(ROOT / "embeddings-smoke.npz"),
                            "--limit", "50", "--device", "cpu"], check=True)
print("50-prompt encoder check passed. Full embeddings are the next cell.")


## 5. Full embeddings, then the same evaluation

Optional GPU acceleration: change the runtime to T4 and rerun the setup/config cells if desired. The same command resumes after a disconnect. Do not run two writers against the same checkpoint directory. Leave `model`, revision, batch size, and truncation unchanged when resuming; use a new output filename for a new configuration.


In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Embedding device:", DEVICE)
EMBEDDINGS = ROOT / "embeddings.npz"
subprocess.run(embed_base + ["--output", str(EMBEDDINGS), "--device", DEVICE], check=True)
print("Checkpointed embeddings:", EMBEDDINGS)


In [ ]:
EMBEDDED_REPORT = ROOT / "cpu-embedded"
subprocess.run([sys.executable, "-m", "tine", "run", "--input", str(DATASET),
                "--frontier", FRONTIER, "--output", str(EMBEDDED_REPORT),
                "--embeddings", str(EMBEDDINGS), "--epsilon", str(EPSILON),
                "--delta", str(DELTA), "--seed", str(SEED)], check=True)
print("Embedded artifacts:", sorted(str(p) for p in EMBEDDED_REPORT.iterdir()))
print("Keep the entire Drive tine directory, especially reports, config, manifests, and embedding checkpoints.")


## Finish

Inspect both report directories on Drive. Compare their held-out metrics and calibration outcomes; do not pick a threshold by inspecting test results. Share the generated reports and manifests for review. A conservative frontier-only result is a valid outcome when sample size or predictor quality is insufficient.

No GPU runtime was executed or model downloaded in the cloud development environment. The notebook's real Colab encoder/download path must be validated when you run it. After artifacts are saved, use **Runtime → Disconnect and delete runtime** so paid compute does not remain idle. Do not deploy a proxy from Colab.
